In [1]:
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import locale
import seaborn as sns
import matplotlib.pyplot as plt
import re
import pyproj
import folium

In [2]:
#Filtros de fechas

fecha = '20260506'

matriz de distancia

In [3]:
matriz_distancia = pd.read_csv(f'Z:/01 base_datos/06 matriz_distancia_FMS/{fecha}_matriz distancias.csv', encoding='latin')

matriz_distancia.head(3)

,ï»¿Tipo de Servicio,Id LÃ­nea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,PosiciÃ³n,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52845.0,247A05_TM,247A05_Br. La Esperanza II,0.0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52806.0,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138.0,595258.0,521149.0,NaN
2,URBANO,10184,740,1,2,338.0,10415.0,740_V1,Circular,52365.0,068A05_TM,068A05_Liceo SalomÃ³n Sabio,425.0,595014.0,520997.0,NaN


In [4]:
matriz_distancia['Id Ruta'] = (
    matriz_distancia['Id Ruta']
    .fillna(0)                 # reemplaza nulos por 0
    .astype(str)               # convierte a texto (por si hay mezcla)
    .str.strip()               # elimina espacios
    .replace('', '0')          # reemplaza vacíos tipo ''
    .pipe(pd.to_numeric, errors='coerce')  # convierte a número
    .fillna(0)                 # por si algo falló
    .astype(int)               # entero final
)

matriz_distancia.head()

,ï»¿Tipo de Servicio,Id LÃ­nea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,PosiciÃ³n,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52845.0,247A05_TM,247A05_Br. La Esperanza II,0.0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52806.0,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138.0,595258.0,521149.0,NaN
2,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52365.0,068A05_TM,068A05_Liceo SalomÃ³n Sabio,425.0,595014.0,520997.0,NaN
3,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52442.0,110A05_TM,110A05_Br. Sabana del Dorado,686.0,594933.0,520819.0,NaN
4,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52369.0,070A05_TM,070A05_Br. Sabana del Dorado,964.0,595091.0,520593.0,NaN


In [5]:
# 🔹 Limpiar espacios
matriz_distancia['PosiciÃ³n'] = (
    matriz_distancia['PosiciÃ³n']
    .astype(str)
    .str.strip()
)

# 🔹 Filtrar inválidos
matriz_distancia = matriz_distancia[
    matriz_distancia['PosiciÃ³n'].notna() &
    (matriz_distancia['PosiciÃ³n'] != '') &
    (matriz_distancia['PosiciÃ³n'].str.lower() != 'nan')
]

# 🔹 Convertir a numérico
matriz_distancia['PosiciÃ³n'] = pd.to_numeric(
    matriz_distancia['PosiciÃ³n'],
    errors='coerce'
)

# 🔹 Eliminar NaN generados
matriz_distancia = matriz_distancia[
    matriz_distancia['PosiciÃ³n'].notna()
]

# 🔹 Convertir a entero
matriz_distancia['PosiciÃ³n'] = (
    matriz_distancia['PosiciÃ³n']
    .astype(float)
    .astype(int)
)

matriz_distancia.head()

,ï»¿Tipo de Servicio,Id LÃ­nea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,PosiciÃ³n,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52845.0,247A05_TM,247A05_Br. La Esperanza II,0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52806.0,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138,595258.0,521149.0,NaN
2,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52365.0,068A05_TM,068A05_Liceo SalomÃ³n Sabio,425,595014.0,520997.0,NaN
3,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52442.0,110A05_TM,110A05_Br. Sabana del Dorado,686,594933.0,520819.0,NaN
4,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52369.0,070A05_TM,070A05_Br. Sabana del Dorado,964,595091.0,520593.0,NaN


In [6]:
# 🔹 Limpiar espacios
matriz_distancia['Id Nodo'] = (
    matriz_distancia['Id Nodo']
    .astype(str)
    .str.strip()
)

# 🔹 Filtrar inválidos
matriz_distancia = matriz_distancia[
    matriz_distancia['Id Nodo'].notna() &
    (matriz_distancia['Id Nodo'] != '') &
    (matriz_distancia['Id Nodo'].str.lower() != 'nan')
]

# 🔹 Convertir a numérico
matriz_distancia['Id Nodo'] = pd.to_numeric(
    matriz_distancia['Id Nodo'],
    errors='coerce'
)

# 🔹 Eliminar NaN generados
matriz_distancia = matriz_distancia[
    matriz_distancia['Id Nodo'].notna()
]

# 🔹 Convertir a entero
matriz_distancia['Id Nodo'] = (
    matriz_distancia['Id Nodo']
    .astype(float)
    .astype(int)
)

matriz_distancia.head()

,ï»¿Tipo de Servicio,Id LÃ­nea,LÃ­nea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,PosiciÃ³n,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52845,247A05_TM,247A05_Br. La Esperanza II,0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52806,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138,595258.0,521149.0,NaN
2,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52365,068A05_TM,068A05_Liceo SalomÃ³n Sabio,425,595014.0,520997.0,NaN
3,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52442,110A05_TM,110A05_Br. Sabana del Dorado,686,594933.0,520819.0,NaN
4,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52369,070A05_TM,070A05_Br. Sabana del Dorado,964,595091.0,520593.0,NaN


In [7]:
# 🔹 Cambiar nombres de columnas
matriz_distancia = matriz_distancia.rename(columns={
    'Id LÃ­nea': 'Id Linea',
    'LÃ­nea': 'Linea',
    'PosiciÃ³n': 'Posicion'
})

matriz_distancia.head()

,ï»¿Tipo de Servicio,Id Linea,Linea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posicion,Coordenada X,Coordenada Y,Atributos
0,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52845,247A05_TM,247A05_Br. La Esperanza II,0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM..."
1,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52806,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138,595258.0,521149.0,NaN
2,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52365,068A05_TM,068A05_Liceo SalomÃ³n Sabio,425,595014.0,520997.0,NaN
3,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52442,110A05_TM,110A05_Br. Sabana del Dorado,686,594933.0,520819.0,NaN
4,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52369,070A05_TM,070A05_Br. Sabana del Dorado,964,595091.0,520593.0,NaN


In [8]:
# Ordenar por Id Linea, Linea y Posicion
matriz_distancia = matriz_distancia.sort_values(
    by=['Id Linea', 'Linea', 'Posicion']
)

# Generar consecutivo iniciando en 1
matriz_distancia['numero_posicion'] = (
    matriz_distancia
    .groupby(['Id Linea', 'Linea'])
    .cumcount() + 1
)

matriz_distancia.head()

,ï»¿Tipo de Servicio,Id Linea,Linea,Configuraciones,ConfiguraciÃ³n Activa,Id SublÃ­nea,Id Ruta,Nombre Ruta,Sentido,Id Nodo,Etiqueta Nodo,Nombre Nodo,Posicion,Coordenada X,Coordenada Y,Atributos,numero_posicion
0,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52845,247A05_TM,247A05_Br. La Esperanza II,0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",1
169,URBANO,10184,740,2,2,2521.0,12774,740_V2,Circular,52845,247A05_TM,247A05_Br. La Esperanza II,0,595348.0,521193.0,"SinÃ³ptico, Horario GOAL, Punto de control (TM...",2
1,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52806,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138,595258.0,521149.0,NaN,3
170,URBANO,10184,740,2,2,2521.0,12774,740_V2,Circular,52806,222A05_TM,222A05_Hospital de EngativÃ¡ EmaÃºs,138,595258.0,521149.0,NaN,4
2,URBANO,10184,740,1,2,338.0,10415,740_V1,Circular,52365,068A05_TM,068A05_Liceo SalomÃ³n Sabio,425,595014.0,520997.0,NaN,5


Rutas

In [9]:
rutas = pd.DataFrame({
    'Id Linea': [
        10311, 10350, 10550, 10266, 10342, 10360, 10354, 10690, 10331,
        10264, 10689, 10325, 10305, 10688, 10273, 10244, 10261, 10204,
        10691, 10196, 10184, 10310, 10304, 10194, 10232, 10292, 10551,
        10282, 10339
    ],
    'Nombre Linea': [
        'DA213', 'SE14', 'P500', 'DD204', 'KB309', '5-abr', '16-may', 'DL219', '12',
        '614', 'DH216', '576', 'DD212', 'BD237', '142', 'SE10', '466', '402',
        'DA218', 'E25', '740', 'C101', '806', '539', '359', '577', 'KL307',
        'DH209', 'C25'
    ]
})

rutas.head()

,Id Linea,Nombre Linea
0,10311,DA213
1,10350,SE14
2,10550,P500
3,10266,DD204
4,10342,KB309


Indice de regularidad por parada

In [10]:
#IRP

irp = pd.read_excel(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/IRP/IRP_diario/{fecha}_IRP_FMS.xlsx')

irp.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Desviacion_Estandar,Coeficiente_Variacion,TipoDia,TextoTipoDia,TipoHora,Promedio_Ruta_En_El_Dia,Desviacion_Estandar_Ruta_En_El_Dia,Coeficiente_Variacion_Ruta_En_El_Dia,Intervalo_Teorico,TP26
0,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,3.96037,54.854939,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,NaN
1,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,3.96037,54.854939,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,NaN
2,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,2,1,1,...,3.96037,54.854939,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,Negativo
3,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,3,1,1,...,3.96037,54.854939,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,Negativo
4,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,4,1,1,...,3.96037,54.854939,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,Negativo


In [11]:
# Dividir la columna  en partes usando ':', y seleccionar la primera parte (horas)
irp['franja'] = irp['Hora Teórica'].str.split(':').str[0]

# Convertir la columna 'franja' a tipo entero
irp['franja'] = pd.to_numeric(irp['franja'], errors='coerce').astype('Int64')

irp.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Coeficiente_Variacion,TipoDia,TextoTipoDia,TipoHora,Promedio_Ruta_En_El_Dia,Desviacion_Estandar_Ruta_En_El_Dia,Coeficiente_Variacion_Ruta_En_El_Dia,Intervalo_Teorico,TP26,franja
0,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,54.854939,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,NaN,3
1,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,54.854939,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,NaN,3
2,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,2,1,1,...,54.854939,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,Negativo,3
3,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,3,1,1,...,54.854939,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,Negativo,3
4,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,4,1,1,...,54.854939,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,Negativo,3


In [12]:
#Llevar tipo día a iri

def calcular_posicion(linea, ruta, nodo):
    
    filtro = (
        (matriz_distancia['Id Linea'] == linea) &
        (matriz_distancia['Id Ruta'] == ruta) &
        (matriz_distancia['Id Nodo'] == nodo)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not matriz_distancia.loc[filtro].empty:
        # Obtener el primer valor
        Tipo_día = matriz_distancia.loc[filtro, 'Posicion'].iloc[0]
        return Tipo_día  if not pd.isna(Tipo_día ) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
irp['Posicion_parada'] = irp.apply(
    lambda row: calcular_posicion(
        row['Id Línea'],
        row['Id Ruta'],
        row['Id Nodo']
    ),
    axis=1
)

irp.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,TipoDia,TextoTipoDia,TipoHora,Promedio_Ruta_En_El_Dia,Desviacion_Estandar_Ruta_En_El_Dia,Coeficiente_Variacion_Ruta_En_El_Dia,Intervalo_Teorico,TP26,franja,Posicion_parada
0,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,NaN,3,15886
1,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,NaN,3,5579
2,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,2,1,1,...,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,Negativo,3,5579
3,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,3,1,1,...,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,Negativo,3,5579
4,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,4,1,1,...,1,Hábil,Valle,10.335012,14.378965,139.128667,7.5,Negativo,3,5579


In [13]:
#Llevar tipo día a iri

def calcular_posicion(linea, ruta, nodo):
    
    filtro = (
        (matriz_distancia['Id Linea'] == linea) &
        (matriz_distancia['Id Ruta'] == ruta) &
        (matriz_distancia['Id Nodo'] == nodo)
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not matriz_distancia.loc[filtro].empty:
        # Obtener el primer valor
        Tipo_día = matriz_distancia.loc[filtro, 'numero_posicion'].iloc[0]
        return Tipo_día  if not pd.isna(Tipo_día ) else None  
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
irp['numero_posicion'] = irp.apply(
    lambda row: calcular_posicion(
        row['Id Línea'],
        row['Id Ruta'],
        row['Id Nodo']
    ),
    axis=1
)

irp.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,TextoTipoDia,TipoHora,Promedio_Ruta_En_El_Dia,Desviacion_Estandar_Ruta_En_El_Dia,Coeficiente_Variacion_Ruta_En_El_Dia,Intervalo_Teorico,TP26,franja,Posicion_parada,numero_posicion
0,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,Hábil,Valle,10.335012,14.378965,139.128667,7.5,NaN,3,15886,81
1,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,Hábil,Valle,10.335012,14.378965,139.128667,7.5,NaN,3,5579,32
2,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,2,1,1,...,Hábil,Valle,10.335012,14.378965,139.128667,7.5,Negativo,3,5579,32
3,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,3,1,1,...,Hábil,Valle,10.335012,14.378965,139.128667,7.5,Negativo,3,5579,32
4,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,4,1,1,...,Hábil,Valle,10.335012,14.378965,139.128667,7.5,Negativo,3,5579,32


In [14]:
#Ordenar dataframe correctamente
irp = irp.sort_values(
    by=[
        'Id Línea',
        'Id Ruta',
        'Tabla',
        'Id Viaje',
        'Posicion_parada',
        'Hora Teórica_minutos'
    ],
    ascending=[True, True, True, True, True, True]
)

# Generar diferencia consecutiva
irp['diferencia_metros'] = (
    irp.groupby(
        ['Id Línea', 'Id Ruta', 'Tabla', 'Id Viaje']
    )['Posicion_parada']
    .diff()
    .fillna(0)
)

# Convertir a entero si deseas
irp['diferencia_metros'] = irp['diferencia_metros'].astype(int)

irp.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,TipoHora,Promedio_Ruta_En_El_Dia,Desviacion_Estandar_Ruta_En_El_Dia,Coeficiente_Variacion_Ruta_En_El_Dia,Intervalo_Teorico,TP26,franja,Posicion_parada,numero_posicion,diferencia_metros
109,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,Valle,10.335012,14.378965,139.128667,7.5,NaN,3,0,2,0
2157,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,Pico,10.335012,14.378965,139.128667,7.5,Positivo,6,0,2,0
99,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,Valle,10.335012,14.378965,139.128667,7.5,NaN,3,138,4,138
7,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,Valle,10.335012,14.378965,139.128667,7.5,NaN,3,425,6,287
42,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,Valle,10.335012,14.378965,139.128667,7.5,NaN,3,686,8,261


In [15]:
# Ordenar correctamente antes de calcular diferencias
irp = irp.sort_values(
    by=[
        'Id Línea',
        'Id Ruta',
        'Tabla',
        'Id Viaje',
        'Hora Teórica_minutos'
    ],
    ascending=True
)

# Diferencia consecutiva de Hora Teórica_minutos
irp['diferencia_minutos'] = (
    irp.groupby(
        ['Id Línea', 'Id Ruta', 'Tabla', 'Id Viaje']
    )['Hora Teórica_minutos']
    .diff()
    .fillna(0)
)

# Convertir a entero
irp['diferencia_minutos'] = (
    irp['diferencia_minutos']
    .round(0)
    .astype(int)
)

irp.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Promedio_Ruta_En_El_Dia,Desviacion_Estandar_Ruta_En_El_Dia,Coeficiente_Variacion_Ruta_En_El_Dia,Intervalo_Teorico,TP26,franja,Posicion_parada,numero_posicion,diferencia_metros,diferencia_minutos
109,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,10.335012,14.378965,139.128667,7.5,NaN,3,0,2,0,0
99,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,10.335012,14.378965,139.128667,7.5,NaN,3,138,4,138,0
7,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,10.335012,14.378965,139.128667,7.5,NaN,3,425,6,287,1
42,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,10.335012,14.378965,139.128667,7.5,NaN,3,686,8,261,1
17,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,10.335012,14.378965,139.128667,7.5,NaN,3,964,10,278,1


In [16]:
irp['velocidad_kmh_prog'] = np.where(
    irp['diferencia_minutos'] > 0,
    (irp['diferencia_metros'] / 1000) /
    (irp['diferencia_minutos'] / 60),
    np.nan
)

irp.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Desviacion_Estandar_Ruta_En_El_Dia,Coeficiente_Variacion_Ruta_En_El_Dia,Intervalo_Teorico,TP26,franja,Posicion_parada,numero_posicion,diferencia_metros,diferencia_minutos,velocidad_kmh_prog
109,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,14.378965,139.128667,7.5,NaN,3,0,2,0,0,NaN
99,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,14.378965,139.128667,7.5,NaN,3,138,4,138,0,NaN
7,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,14.378965,139.128667,7.5,NaN,3,425,6,287,1,17.22
42,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,14.378965,139.128667,7.5,NaN,3,686,8,261,1,15.66
17,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,14.378965,139.128667,7.5,NaN,3,964,10,278,1,16.68


In [17]:
# Ordenar correctamente antes de calcular diferencias
irp = irp.sort_values(
    by=[
        'Id Línea',
        'Id Ruta',
        'Tabla',
        'Id Viaje',
        'Hora Referencia_minutos'
    ],
    ascending=True
)

# Diferencia consecutiva de Hora Referencia_minutos
irp['diferencia_minutos_ref'] = (
    irp.groupby(
        ['Id Línea', 'Id Ruta', 'Tabla', 'Id Viaje']
    )['Hora Referencia_minutos']
    .diff()
    .fillna(0)
)

# Convertir a entero
irp['diferencia_minutos_ref'] = (
    irp['diferencia_minutos_ref']
    .round(0)
    .astype(int)
)

irp.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Coeficiente_Variacion_Ruta_En_El_Dia,Intervalo_Teorico,TP26,franja,Posicion_parada,numero_posicion,diferencia_metros,diferencia_minutos,velocidad_kmh_prog,diferencia_minutos_ref
109,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,139.128667,7.5,NaN,3,0,2,0,0,NaN,0
99,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,139.128667,7.5,NaN,3,138,4,138,0,NaN,0
7,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,139.128667,7.5,NaN,3,425,6,287,1,17.22,1
42,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,139.128667,7.5,NaN,3,686,8,261,1,15.66,1
17,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,139.128667,7.5,NaN,3,964,10,278,1,16.68,1


In [18]:
irp['velocidad_kmh_ref'] = np.where(
    irp['diferencia_minutos_ref'] > 0,
    (irp['diferencia_metros'] / 1000) /
    (irp['diferencia_minutos_ref'] / 60),
    np.nan
)

irp.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,Intervalo_Teorico,TP26,franja,Posicion_parada,numero_posicion,diferencia_metros,diferencia_minutos,velocidad_kmh_prog,diferencia_minutos_ref,velocidad_kmh_ref
109,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,7.5,NaN,3,0,2,0,0,NaN,0,NaN
99,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,7.5,NaN,3,138,4,138,0,NaN,0,NaN
7,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,7.5,NaN,3,425,6,287,1,17.22,1,17.22
42,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,7.5,NaN,3,686,8,261,1,15.66,1,15.66
17,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,7.5,NaN,3,964,10,278,1,16.68,1,16.68


In [19]:
# Ordenar correctamente antes de calcular diferencias
irp = irp.sort_values(
    by=[
        'Id Línea',
        'Id Ruta',
        'Tabla',
        'Id Viaje',
        'Hora Llegada_minutos'
    ],
    ascending=True
)

# Diferencia consecutiva de Hora Referencia_minutos
irp['diferencia_minutos_llegada'] = (
    irp.groupby(
        ['Id Línea', 'Id Ruta', 'Tabla', 'Id Viaje']
    )['Hora Llegada_minutos']
    .diff()
    .fillna(0)
)

# Convertir a entero
irp['diferencia_minutos_llegada'] = (
    irp['diferencia_minutos_llegada']
    .round(0)
    .astype(int)
)

irp.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,TP26,franja,Posicion_parada,numero_posicion,diferencia_metros,diferencia_minutos,velocidad_kmh_prog,diferencia_minutos_ref,velocidad_kmh_ref,diferencia_minutos_llegada
109,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,NaN,3,0,2,0,0,NaN,0,NaN,0
99,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,NaN,3,138,4,138,0,NaN,0,NaN,0
7,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,NaN,3,425,6,287,1,17.22,1,17.22,1
42,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,NaN,3,686,8,261,1,15.66,1,15.66,1
17,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,NaN,3,964,10,278,1,16.68,1,16.68,1


In [20]:
irp['velocidad_kmh_real'] = np.where(
    irp['diferencia_minutos_llegada'] > 0,
    (irp['diferencia_metros'] / 1000) /
    (irp['diferencia_minutos_llegada'] / 60),
    np.nan
)

irp.head()

,Fecha,Concesión,Concesionario de Operación,Id Línea,Línea,Id Ruta,Ruta,Tabla,Viaje Linea,Orden Viaje,...,franja,Posicion_parada,numero_posicion,diferencia_metros,diferencia_minutos,velocidad_kmh_prog,diferencia_minutos_ref,velocidad_kmh_ref,diferencia_minutos_llegada,velocidad_kmh_real
109,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,3,0,2,0,0,NaN,0,NaN,0,NaN
99,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,3,138,4,138,0,NaN,0,NaN,0,NaN
7,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,3,425,6,287,1,17.22,1,17.22,1,17.22
42,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,3,686,8,261,1,15.66,1,15.66,1,15.66
17,2026-05-06,ENGATIVA ZN,(105) GMOVIL ENGATIVA,10184,740,12774,740_V2,1,1,1,...,3,964,10,278,1,16.68,1,16.68,1,16.68


In [21]:
irp.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Velocidad/{fecha}_velocidad.csv', index=False, sep=';')